# Assignment 2 — Delta Lake: Incremental Processing with `MERGE` (SCD Type 1 & Type 2)

**Objective:** Perform incremental data processing using Delta Lake.

| # | Step | Section |
|---|------|---------|
| 1 | Load dataset into a Delta table | 2–3 |
| 2 | Basic cleaning (handle nulls, remove duplicates) | 2 |
| 3 | Create a second dataset simulating new / incremental data | 4 |
| 4 | Apply `MERGE` to update existing and insert new records | 5 (SCD1), 6 (SCD2) |
| 5 | Validate results (row count, duplicates) | 8 |
| 6 | Display final dataset and summary | 9 |

**Extras included:** `WHEN NOT MATCHED BY SOURCE`, the SQL flavour of `MERGE`,
`DESCRIBE HISTORY` and time travel.

---

### Before you run this

This notebook needs a **JDK (11 or 17)** and downloads the Delta Lake jars from Maven
on the very first `getOrCreate()` — so the first run takes 1–3 minutes and needs internet.
See `README.md` → *Environment setup* if the Spark session fails to start.

## 0. Environment check and Spark session

In [181]:
import os, shutil, subprocess, sys
from pathlib import Path

# --- Java check --------------------------------------------------------
try:
    out = subprocess.run(["java", "-version"], capture_output=True, text=True)
    print((out.stderr or out.stdout).strip().splitlines()[0])
except FileNotFoundError:
    print("!! Java not found on PATH. Install a JDK 17 and set JAVA_HOME. See README.")

print("JAVA_HOME =", os.environ.get("JAVA_HOME", "(not set)"))
print("python    =", sys.version.split()[0])

openjdk version "21.0.11" 2026-04-21 LTS
JAVA_HOME = C:\Users\hp\AppData\Local\Programs\Microsoft\jdk-21\jdk-21.0.11+10
python    = 3.12.10


In [182]:
import pyspark
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import StringType, DateType

print("pyspark  :", pyspark.__version__)

# On Windows, make sure the worker uses the same interpreter as the notebook.
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

pyspark  : 3.5.1


In [183]:
from pyspark.sql.types import StructType, StructField

# Read every column as a STRING, then cast deliberately.
# Automatic type inference is convenient but unpredictable here: it would turn
# `phone` into a bigint (dropping any leading zero) and may auto-parse
# `updated_at` into a date, which then breaks the explicit to_date() call below.
# An explicit schema means no surprises.
CUSTOMER_SCHEMA = StructType([
    StructField("customer_id", StringType(), True),
    StructField("name",        StringType(), True),
    StructField("email",       StringType(), True),
    StructField("city",        StringType(), True),
    StructField("segment",     StringType(), True),
    StructField("phone",       StringType(), True),
    StructField("updated_at",  StringType(), True),
])

print("Schema defined for the customer datasets.")

Schema defined for the customer datasets.


In [184]:
# --- Project paths -----------------------------------------------------
NB_DIR = Path.cwd()
PROJECT_ROOT = NB_DIR if (NB_DIR / "data").is_dir() else NB_DIR.parent

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "output"
LAKE_DIR = OUTPUT_DIR / "delta"
OUTPUT_DIR.mkdir(exist_ok=True)
LAKE_DIR.mkdir(parents=True, exist_ok=True)


def spark_path(p) -> str:
    """Spark wants a URI. Path.as_uri() gets Windows drive letters right."""
    return Path(p).resolve().as_uri()


SCD1_PATH = LAKE_DIR / "customer_scd1"
SCD2_PATH = LAKE_DIR / "customer_scd2"
SYNC_PATH = LAKE_DIR / "customer_sync_demo"

print("project root :", PROJECT_ROOT)
print("delta lake   :", LAKE_DIR)

project root : c:\Users\hp\Desktop\CelebalAssignments\Assignment7
delta lake   : c:\Users\hp\Desktop\CelebalAssignments\Assignment7\output\delta


In [185]:
# --- Start Spark with the Delta extensions -----------------------------
# configure_spark_with_delta_pip() adds the matching io.delta jars. The FIRST run
# downloads them from Maven Central (needs internet); later runs use the ivy cache.

builder = (
    SparkSession.builder
    .appName("DeltaLakeIncrementalAssignment")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    # small local dataset -> keep the shuffle tiny, otherwise Spark makes 200 files
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.databricks.delta.snapshotPartitions", "2")
    .config("spark.sql.warehouse.dir", spark_path(OUTPUT_DIR / "spark-warehouse"))
    # legacy parser so 'dd/MM/yyyy'-style strings don't raise on some builds
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("Spark version :", spark.version)
print("Delta enabled :", "io.delta" in spark.conf.get("spark.sql.extensions"))

Spark version : 3.5.1
Delta enabled : True


## 1. Load the source dataset

`data/customer_master.csv` is the initial full load. It contains deliberate problems:
duplicate rows, null emails/phones/cities, and one row with **no `customer_id` at all**.

In [186]:
raw_master = (
    spark.read
    .option("header", True)
    .schema(CUSTOMER_SCHEMA)
    .csv(spark_path(DATA_DIR / "customer_master.csv"))
)

print(f"Raw master rows: {raw_master.count()}")
raw_master.printSchema()
raw_master.show(truncate=False)

Raw master rows: 18
root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- updated_at: string (nullable = true)

+-----------+--------------+------------------------+----------+-----------+----------+----------+
|customer_id|name          |email                   |city      |segment    |phone     |updated_at|
+-----------+--------------+------------------------+----------+-----------+----------+----------+
|C001       |Aarav Sharma  |aarav.sharma@example.com|Mumbai    |Consumer   |9810000001|2024-01-10|
|C002       |Bhavna Mehta  |bhavna.mehta@example.com|Pune      |Corporate  |9810000002|2024-01-10|
|C003       |Chirag Iyer   |chirag.iyer@example.com |Bengaluru |Consumer   |9810000003|2024-01-11|
|C004       |Divya Nair    |divya.nair@example.com  |Kochi     |Home Office|9810000004|2024-01-11|
|C005      

## 2. Basic cleaning — nulls, whitespace and duplicates

In [187]:
# --- 2.1 Identify nulls per column ------------------------------------
null_counts = raw_master.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in raw_master.columns
])
print("Null count per column (BEFORE cleaning):")
null_counts.show()

print(f"Duplicate rows (exact)      : "
      f"{raw_master.count() - raw_master.dropDuplicates().count()}")
print(f"Duplicate customer_id values: "
      f"{raw_master.count() - raw_master.dropDuplicates(['customer_id']).count()}")

Null count per column (BEFORE cleaning):
+-----------+----+-----+----+-------+-----+----------+
|customer_id|name|email|city|segment|phone|updated_at|
+-----------+----+-----+----+-------+-----+----------+
|          1|   0|    1|   1|      0|    1|         0|
+-----------+----+-----+----+-------+-----+----------+

Duplicate rows (exact)      : 2
Duplicate customer_id values: 2


In [188]:
# --- 2.2 Clean ---------------------------------------------------------
TRIM_COLS = ["customer_id", "name", "email", "city", "segment", "phone"]

clean_master = raw_master
for c in TRIM_COLS:
    clean_master = clean_master.withColumn(c, F.trim(F.col(c)))

clean_master = (
    clean_master
    # (a) a record without a business key cannot be merged -> drop it
    .filter(F.col("customer_id").isNotNull() & (F.col("customer_id") != ""))
    # (b) fill the optional attributes so downstream joins don't produce NULLs
    .fillna({"email": "unknown@example.com", "phone": "0000000000",
             "city": "Unknown", "segment": "Unknown"})
    # (c) proper date type for the change timestamp
    .withColumn("updated_at", F.to_date("updated_at", "yyyy-MM-dd"))
    # (d) drop exact duplicate rows
    .dropDuplicates()
)

# (e) enforce one row per customer_id — keep the most recent by updated_at
w = Window.partitionBy("customer_id").orderBy(F.col("updated_at").desc())
clean_master = (
    clean_master
    .withColumn("_rn", F.row_number().over(w))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

print(f"Rows after cleaning: {clean_master.count()}  (raw was {raw_master.count()})")
clean_master.orderBy("customer_id").show(truncate=False)

Rows after cleaning: 15  (raw was 18)
+-----------+--------------+------------------------+----------+-----------+----------+----------+
|customer_id|name          |email                   |city      |segment    |phone     |updated_at|
+-----------+--------------+------------------------+----------+-----------+----------+----------+
|C001       |Aarav Sharma  |aarav.sharma@example.com|Mumbai    |Consumer   |9810000001|2024-01-10|
|C002       |Bhavna Mehta  |bhavna.mehta@example.com|Pune      |Corporate  |9810000002|2024-01-10|
|C003       |Chirag Iyer   |chirag.iyer@example.com |Bengaluru |Consumer   |9810000003|2024-01-11|
|C004       |Divya Nair    |divya.nair@example.com  |Kochi     |Home Office|9810000004|2024-01-11|
|C005       |Esha Khan     |esha.khan@example.com   |Delhi     |Corporate  |0000000000|2024-01-12|
|C006       |Farhan Ali    |farhan.ali@example.com  |Hyderabad |Consumer   |9810000006|2024-01-12|
|C007       |Gita Verma    |unknown@example.com     |Jaipur    |Consume

In [189]:
# --- 2.3 Prove the cleaning worked ------------------------------------
assert clean_master.filter(F.col("customer_id").isNull()).count() == 0, "null keys remain"
assert clean_master.count() == clean_master.dropDuplicates(["customer_id"]).count(), \
    "duplicate customer_id remains"
print("Cleaning checks passed: no null keys, no duplicate customer_id.")

Cleaning checks passed: no null keys, no duplicate customer_id.


## 3. Write the cleaned data into Delta tables

Two target tables are created from the same cleaned snapshot:

* **`customer_scd1`** — *Type 1*: overwrite the attribute, keep no history.
* **`customer_scd2`** — *Type 2*: keep every version, flagged with
  `start_date` / `end_date` / `is_current`.

Both are also registered in the metastore so the SQL `MERGE` syntax works later.

In [190]:
# Start from a clean slate so the notebook is re-runnable end to end.
for p in (SCD1_PATH, SCD2_PATH, SYNC_PATH):
    if p.exists():
        shutil.rmtree(p)

for t in ("customer_scd1", "customer_scd2", "customer_sync_demo"):
    spark.sql(f"DROP TABLE IF EXISTS {t}")

print("Old Delta tables removed.")

Old Delta tables removed.


In [191]:
# --- 3.1 SCD Type 1 target --------------------------------------------
(clean_master
 .write
 .format("delta")
 .mode("overwrite")
 .save(spark_path(SCD1_PATH)))

spark.sql(f"CREATE TABLE customer_scd1 USING DELTA LOCATION '{spark_path(SCD1_PATH)}'")

print(f"customer_scd1 written -> {SCD1_PATH}")
spark.table("customer_scd1").orderBy("customer_id").show(5, truncate=False)

customer_scd1 written -> c:\Users\hp\Desktop\CelebalAssignments\Assignment7\output\delta\customer_scd1
+-----------+------------+------------------------+---------+-----------+----------+----------+
|customer_id|name        |email                   |city     |segment    |phone     |updated_at|
+-----------+------------+------------------------+---------+-----------+----------+----------+
|C001       |Aarav Sharma|aarav.sharma@example.com|Mumbai   |Consumer   |9810000001|2024-01-10|
|C002       |Bhavna Mehta|bhavna.mehta@example.com|Pune     |Corporate  |9810000002|2024-01-10|
|C003       |Chirag Iyer |chirag.iyer@example.com |Bengaluru|Consumer   |9810000003|2024-01-11|
|C004       |Divya Nair  |divya.nair@example.com  |Kochi    |Home Office|9810000004|2024-01-11|
|C005       |Esha Khan   |esha.khan@example.com   |Delhi    |Corporate  |0000000000|2024-01-12|
+-----------+------------+------------------------+---------+-----------+----------+----------+
only showing top 5 rows



In [192]:
# --- 3.2 SCD Type 2 target (with history-tracking columns) -------------
scd2_seed = (
    clean_master
    .withColumn("start_date", F.col("updated_at"))
    .withColumn("end_date", F.lit(None).cast(DateType()))
    .withColumn("is_current", F.lit(True))
)

(scd2_seed
 .write
 .format("delta")
 .mode("overwrite")
 .save(spark_path(SCD2_PATH)))

spark.sql(f"CREATE TABLE customer_scd2 USING DELTA LOCATION '{spark_path(SCD2_PATH)}'")

print(f"customer_scd2 written -> {SCD2_PATH}")
spark.table("customer_scd2").orderBy("customer_id").show(5, truncate=False)

customer_scd2 written -> c:\Users\hp\Desktop\CelebalAssignments\Assignment7\output\delta\customer_scd2
+-----------+------------+------------------------+---------+-----------+----------+----------+----------+--------+----------+
|customer_id|name        |email                   |city     |segment    |phone     |updated_at|start_date|end_date|is_current|
+-----------+------------+------------------------+---------+-----------+----------+----------+----------+--------+----------+
|C001       |Aarav Sharma|aarav.sharma@example.com|Mumbai   |Consumer   |9810000001|2024-01-10|2024-01-10|NULL    |true      |
|C002       |Bhavna Mehta|bhavna.mehta@example.com|Pune     |Corporate  |9810000002|2024-01-10|2024-01-10|NULL    |true      |
|C003       |Chirag Iyer |chirag.iyer@example.com |Bengaluru|Consumer   |9810000003|2024-01-11|2024-01-11|NULL    |true      |
|C004       |Divya Nair  |divya.nair@example.com  |Kochi    |Home Office|9810000004|2024-01-11|2024-01-11|NULL    |true      |
|C005   

In [193]:
# Confirm these really are Delta tables (a _delta_log folder = transaction log)
print(sorted(os.listdir(SCD1_PATH))[:6])
spark.sql("DESCRIBE DETAIL customer_scd1").select(
    "format", "numFiles", "sizeInBytes", "location").show(truncate=False)

['.part-00000-c87596d7-464a-4069-8776-3a35279f686c-c000.snappy.parquet.crc', '_delta_log', 'part-00000-c87596d7-464a-4069-8776-3a35279f686c-c000.snappy.parquet']
+------+--------+-----------+-----------------------------------------------------------------------------------+
|format|numFiles|sizeInBytes|location                                                                           |
+------+--------+-----------+-----------------------------------------------------------------------------------+
|delta |1       |2940       |file:/C:/Users/hp/Desktop/CelebalAssignments/Assignment7/output/delta/customer_scd1|
+------+--------+-----------+-----------------------------------------------------------------------------------+



## 4. The incremental batch

`data/customer_incremental.csv` simulates the next day's feed. It deliberately mixes
five situations, which is what makes the merge interesting:

| Situation | Example |
|---|---|
| Attribute changed | `C002` moved Pune → Bengaluru |
| Attribute changed | `C005`, `C009`, `C012` |
| **No-op** (identical to what's stored) | `C001` |
| Brand new customer | `C016`, `C017`, `C018` |
| **Out-of-order / late-arriving** older row | second `C002` row dated 2024-02-28 |
| Exact duplicate inside the batch | second `C017` row |

In [194]:
raw_incr = (
    spark.read.option("header", True).schema(CUSTOMER_SCHEMA)
    .csv(spark_path(DATA_DIR / "customer_incremental.csv"))
)

print(f"Raw incremental rows: {raw_incr.count()}")
raw_incr.show(truncate=False)

Raw incremental rows: 10
+-----------+--------------+------------------------+---------+-----------+----------+----------+
|customer_id|name          |email                   |city     |segment    |phone     |updated_at|
+-----------+--------------+------------------------+---------+-----------+----------+----------+
|C002       |Bhavna Mehta  |bhavna.mehta@example.com|Bengaluru|Corporate  |9810000002|2024-03-02|
|C005       |Esha Khan     |esha.khan@corp.com      |Delhi    |Home Office|9820000005|2024-03-02|
|C009       |Ishita Rao    |ishita.rao@example.com  |Goa      |Home Office|9810000009|2024-03-03|
|C012       |Lakshay Bhatia|lakshay.b@example.com   |Noida    |Corporate  |9810000012|2024-03-03|
|C001       |Aarav Sharma  |aarav.sharma@example.com|Mumbai   |Consumer   |9810000001|2024-03-04|
|C016       |Priya Deshmukh|priya.d@example.com     |Surat    |Consumer   |9810000016|2024-03-04|
|C017       |Rahul Sinha   |rahul.sinha@example.com |Lucknow  |Corporate  |9810000017|2024-03

### 4.1 De-duplicate the batch *before* merging

This step is not optional. Delta raises
`UnsupportedOperationException: Cannot perform Merge as multiple source rows matched...`
if two source rows match the same target row — the update would be ambiguous.

A `row_number()` window ordered by `updated_at DESC` keeps only the newest row per
customer, which fixes the duplicate **and** the out-of-order record in one pass.

In [195]:
incr = raw_incr
for c in TRIM_COLS:
    incr = incr.withColumn(c, F.trim(F.col(c)))

incr = (
    incr
    .filter(F.col("customer_id").isNotNull() & (F.col("customer_id") != ""))
    .fillna({"email": "unknown@example.com", "phone": "0000000000",
             "city": "Unknown", "segment": "Unknown"})
    .withColumn("updated_at", F.to_date("updated_at", "yyyy-MM-dd"))
    .dropDuplicates()
)

w_incr = Window.partitionBy("customer_id").orderBy(F.col("updated_at").desc())
updates = (
    incr.withColumn("_rn", F.row_number().over(w_incr))
        .filter(F.col("_rn") == 1)
        .drop("_rn")
)

print(f"Incremental rows: {raw_incr.count()} raw -> {updates.count()} after de-dup")
updates.orderBy("customer_id").show(truncate=False)

assert updates.count() == updates.dropDuplicates(["customer_id"]).count()
print("One row per customer_id in the source batch — safe to MERGE.")

Incremental rows: 10 raw -> 8 after de-dup
+-----------+--------------+------------------------+---------+-----------+----------+----------+
|customer_id|name          |email                   |city     |segment    |phone     |updated_at|
+-----------+--------------+------------------------+---------+-----------+----------+----------+
|C001       |Aarav Sharma  |aarav.sharma@example.com|Mumbai   |Consumer   |9810000001|2024-03-04|
|C002       |Bhavna Mehta  |bhavna.mehta@example.com|Bengaluru|Corporate  |9810000002|2024-03-02|
|C005       |Esha Khan     |esha.khan@corp.com      |Delhi    |Home Office|9820000005|2024-03-02|
|C009       |Ishita Rao    |ishita.rao@example.com  |Goa      |Home Office|9810000009|2024-03-03|
|C012       |Lakshay Bhatia|lakshay.b@example.com   |Noida    |Corporate  |9810000012|2024-03-03|
|C016       |Priya Deshmukh|priya.d@example.com     |Surat    |Consumer   |9810000016|2024-03-04|
|C017       |Rahul Sinha   |rahul.sinha@example.com |Lucknow  |Corporate  |

In [196]:
# What is actually going to happen? Classify each incoming row up front.
existing_ids = [r.customer_id for r in
                spark.table("customer_scd1").select("customer_id").collect()]

preview = updates.withColumn(
    "action",
    F.when(F.col("customer_id").isin(existing_ids), F.lit("UPDATE (existing)"))
     .otherwise(F.lit("INSERT (new)"))
)
preview.groupBy("action").count().show()
preview.select("customer_id", "name", "city", "segment", "updated_at", "action") \
       .orderBy("action", "customer_id").show(truncate=False)

+-----------------+-----+
|           action|count|
+-----------------+-----+
|UPDATE (existing)|    5|
|     INSERT (new)|    3|
+-----------------+-----+

+-----------+--------------+---------+-----------+----------+-----------------+
|customer_id|name          |city     |segment    |updated_at|action           |
+-----------+--------------+---------+-----------+----------+-----------------+
|C016       |Priya Deshmukh|Surat    |Consumer   |2024-03-04|INSERT (new)     |
|C017       |Rahul Sinha   |Lucknow  |Corporate  |2024-03-05|INSERT (new)     |
|C018       |Sneha Kapoor  |Bhopal   |Consumer   |2024-03-05|INSERT (new)     |
|C001       |Aarav Sharma  |Mumbai   |Consumer   |2024-03-04|UPDATE (existing)|
|C002       |Bhavna Mehta  |Bengaluru|Corporate  |2024-03-02|UPDATE (existing)|
|C005       |Esha Khan     |Delhi    |Home Office|2024-03-02|UPDATE (existing)|
|C009       |Ishita Rao    |Goa      |Home Office|2024-03-03|UPDATE (existing)|
|C012       |Lakshay Bhatia|Noida    |Corpo

## 5. `MERGE` — SCD Type 1 (upsert, no history)

The classic upsert: **update the row when the key matches, insert it when it doesn't.**

The `condition="s.updated_at > t.updated_at"` on the matched clause is the important
detail — without it a late-arriving old record would happily overwrite newer data.

In [197]:
scd1 = DeltaTable.forPath(spark, spark_path(SCD1_PATH))

print("BEFORE merge:")
before_scd1 = scd1.toDF()

# Materialise the pre-merge state NOW. A Spark DataFrame is lazy, so if these were
# left unevaluated they would silently re-read the table *after* the merge.
before_count = before_scd1.count()
before_ids = {r["customer_id"] for r in before_scd1.select("customer_id").collect()}

print(f"  rows = {before_count}")
before_scd1.filter(F.col("customer_id").isin("C001", "C002", "C005", "C016")) \
           .orderBy("customer_id").show(truncate=False)

BEFORE merge:
  rows = 15
+-----------+------------+------------------------+------+---------+----------+----------+
|customer_id|name        |email                   |city  |segment  |phone     |updated_at|
+-----------+------------+------------------------+------+---------+----------+----------+
|C001       |Aarav Sharma|aarav.sharma@example.com|Mumbai|Consumer |9810000001|2024-01-10|
|C002       |Bhavna Mehta|bhavna.mehta@example.com|Pune  |Corporate|9810000002|2024-01-10|
|C005       |Esha Khan   |esha.khan@example.com   |Delhi |Corporate|0000000000|2024-01-12|
+-----------+------------+------------------------+------+---------+----------+----------+



In [198]:
(scd1.alias("t")
 .merge(
     source=updates.alias("s"),
     condition="t.customer_id = s.customer_id"
 )
 # only overwrite when the incoming record is genuinely newer
 .whenMatchedUpdate(
     condition="s.updated_at > t.updated_at",
     set={
         "name":       "s.name",
         "email":      "s.email",
         "city":       "s.city",
         "segment":    "s.segment",
         "phone":      "s.phone",
         "updated_at": "s.updated_at",
     },
 )
 .whenNotMatchedInsert(
     values={
         "customer_id": "s.customer_id",
         "name":        "s.name",
         "email":       "s.email",
         "city":        "s.city",
         "segment":     "s.segment",
         "phone":       "s.phone",
         "updated_at":  "s.updated_at",
     },
 )
 .execute())

print("SCD Type 1 merge complete.")

SCD Type 1 merge complete.


In [199]:
after_scd1 = spark.table("customer_scd1")
print(f"AFTER merge: rows = {after_scd1.count()} (was {before_count})")
after_scd1.orderBy("customer_id").show(50, truncate=False)

AFTER merge: rows = 18 (was 15)
+-----------+--------------+------------------------+----------+-----------+----------+----------+
|customer_id|name          |email                   |city      |segment    |phone     |updated_at|
+-----------+--------------+------------------------+----------+-----------+----------+----------+
|C001       |Aarav Sharma  |aarav.sharma@example.com|Mumbai    |Consumer   |9810000001|2024-03-04|
|C002       |Bhavna Mehta  |bhavna.mehta@example.com|Bengaluru |Corporate  |9810000002|2024-03-02|
|C003       |Chirag Iyer   |chirag.iyer@example.com |Bengaluru |Consumer   |9810000003|2024-01-11|
|C004       |Divya Nair    |divya.nair@example.com  |Kochi     |Home Office|9810000004|2024-01-11|
|C005       |Esha Khan     |esha.khan@corp.com      |Delhi     |Home Office|9820000005|2024-03-02|
|C006       |Farhan Ali    |farhan.ali@example.com  |Hyderabad |Consumer   |9810000006|2024-01-12|
|C007       |Gita Verma    |unknown@example.com     |Jaipur    |Consumer   |9

In [200]:
# What the operation actually did, straight from the transaction log
(spark.sql("DESCRIBE HISTORY customer_scd1")
      .select("version", "operation", "operationMetrics")
      .show(truncate=False))

+-------+---------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|operation|operationMetrics                                                                                                                                                                                                                                                                                                                         

### 5.1 The same merge in SQL

`whenMatchedUpdateAll()` / `whenNotMatchedInsertAll()` (`UPDATE SET *` / `INSERT *` in SQL)
are the shorthand when the source and target schemas match exactly. Re-running the merge is
a no-op here, which is the point — **`MERGE` is idempotent** for an unchanged source.

In [201]:
updates.createOrReplaceTempView("customer_updates")

spark.sql("""
MERGE INTO customer_scd1 AS t
USING customer_updates AS s
ON t.customer_id = s.customer_id
WHEN MATCHED AND s.updated_at > t.updated_at THEN
  UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *
""")

print("SQL MERGE executed. Row count:", spark.table("customer_scd1").count())

# DESCRIBE HISTORY is a Delta command, not a queryable relation, so pull the
# metrics out with the DataFrame API rather than wrapping it in a subquery.
(spark.sql("DESCRIBE HISTORY customer_scd1")
      .select(
          "version",
          "operation",
          F.col("operationMetrics")["numTargetRowsUpdated"].alias("updated"),
          F.col("operationMetrics")["numTargetRowsInserted"].alias("inserted"),
      )
      .orderBy(F.col("version").desc())
      .limit(3)
      .show(truncate=False))

SQL MERGE executed. Row count: 18
+-------+---------+-------+--------+
|version|operation|updated|inserted|
+-------+---------+-------+--------+
|2      |MERGE    |0      |0       |
|1      |MERGE    |5      |3       |
|0      |WRITE    |NULL   |NULL    |
+-------+---------+-------+--------+



## 6. `MERGE` — SCD Type 2 (full history)

Type 2 never overwrites. When a tracked attribute changes it:

1. **closes** the current row — `is_current = false`, `end_date = <change date>`
2. **inserts** a brand-new row for the same key — `is_current = true`, `end_date = null`

Delta's `MERGE` can only take one action per matched row, so the standard trick is a
**staged source with a `mergeKey` column**:

* every incoming row is staged once with `mergeKey = customer_id` → matches the current
  row → **closes** it
* rows whose attributes actually changed are staged a *second* time with `mergeKey = NULL`
  → can never match → falls through to `WHEN NOT MATCHED` → **inserts** the new version

In [202]:
TRACKED = ["name", "email", "city", "segment", "phone"]

# '<=>' is the null-safe equality operator, so NULL vs NULL counts as "same".
change_condition = " OR ".join([f"NOT (t.{c} <=> s.{c})" for c in TRACKED])
print("Change-detection condition:\n ", change_condition)

Change-detection condition:
  NOT (t.name <=> s.name) OR NOT (t.email <=> s.email) OR NOT (t.city <=> s.city) OR NOT (t.segment <=> s.segment) OR NOT (t.phone <=> s.phone)


In [203]:
scd2 = DeltaTable.forPath(spark, spark_path(SCD2_PATH))
current_rows = scd2.toDF().filter(F.col("is_current") == True)  # noqa: E712

# Which incoming rows differ from the currently-active version?
changed_rows = (
    updates.alias("s")
    .join(current_rows.alias("t"), F.col("s.customer_id") == F.col("t.customer_id"))
    .where(change_condition)
    .where("s.updated_at > t.start_date")   # ignore late-arriving old records
    .select("s.*")
)

print(f"Rows with a real attribute change: {changed_rows.count()}")
changed_rows.select("customer_id", "city", "segment", "email", "updated_at") \
            .orderBy("customer_id").show(truncate=False)

Rows with a real attribute change: 4
+-----------+---------+-----------+------------------------+----------+
|customer_id|city     |segment    |email                   |updated_at|
+-----------+---------+-----------+------------------------+----------+
|C002       |Bengaluru|Corporate  |bhavna.mehta@example.com|2024-03-02|
|C005       |Delhi    |Home Office|esha.khan@corp.com      |2024-03-02|
|C009       |Goa      |Home Office|ishita.rao@example.com  |2024-03-03|
|C012       |Noida    |Corporate  |lakshay.b@example.com   |2024-03-03|
+-----------+---------+-----------+------------------------+----------+



In [204]:
# Stage the source: one pass to CLOSE, a second (mergeKey = NULL) to INSERT.
staged_updates = (
    updates.withColumn("mergeKey", F.col("customer_id"))
    .unionByName(
        changed_rows.withColumn("mergeKey", F.lit(None).cast(StringType()))
    )
)

print(f"Staged source rows: {staged_updates.count()} "
      f"({updates.count()} to match + {changed_rows.count()} to insert)")
staged_updates.select("mergeKey", "customer_id", "city", "segment", "updated_at") \
              .orderBy(F.col("mergeKey").asc_nulls_last(), "customer_id") \
              .show(truncate=False)

Staged source rows: 12 (8 to match + 4 to insert)
+--------+-----------+---------+-----------+----------+
|mergeKey|customer_id|city     |segment    |updated_at|
+--------+-----------+---------+-----------+----------+
|C001    |C001       |Mumbai   |Consumer   |2024-03-04|
|C002    |C002       |Bengaluru|Corporate  |2024-03-02|
|C005    |C005       |Delhi    |Home Office|2024-03-02|
|C009    |C009       |Goa      |Home Office|2024-03-03|
|C012    |C012       |Noida    |Corporate  |2024-03-03|
|C016    |C016       |Surat    |Consumer   |2024-03-04|
|C017    |C017       |Lucknow  |Corporate  |2024-03-05|
|C018    |C018       |Bhopal   |Consumer   |2024-03-05|
|NULL    |C002       |Bengaluru|Corporate  |2024-03-02|
|NULL    |C005       |Delhi    |Home Office|2024-03-02|
|NULL    |C009       |Goa      |Home Office|2024-03-03|
|NULL    |C012       |Noida    |Corporate  |2024-03-03|
+--------+-----------+---------+-----------+----------+



In [205]:
(scd2.alias("t")
 .merge(
     source=staged_updates.alias("s"),
     condition="t.customer_id = s.mergeKey AND t.is_current = true"
 )
 # ---- close out the superseded version
 .whenMatchedUpdate(
     condition=change_condition,
     set={
         "is_current": F.lit(False),
         "end_date":   F.col("s.updated_at"),
     },
 )
 # ---- insert the new version (and brand-new customers)
 .whenNotMatchedInsert(
     values={
         "customer_id": "s.customer_id",
         "name":        "s.name",
         "email":       "s.email",
         "city":        "s.city",
         "segment":     "s.segment",
         "phone":       "s.phone",
         "updated_at":  "s.updated_at",
         "start_date":  "s.updated_at",
         "end_date":    F.lit(None).cast(DateType()),
         "is_current":  F.lit(True),
     },
 )
 .execute())

print("SCD Type 2 merge complete.")

SCD Type 2 merge complete.


In [206]:
scd2_df = spark.table("customer_scd2")
print(f"customer_scd2 rows: {scd2_df.count()}")

print("\n--- Customers that now have HISTORY (more than one version) ---")
multi = (scd2_df.groupBy("customer_id").count()
                .filter("count > 1").select("customer_id"))
(scd2_df.join(multi, "customer_id")
        .orderBy("customer_id", "start_date")
        .select("customer_id", "name", "city", "segment", "email",
                "start_date", "end_date", "is_current")
        .show(truncate=False))

customer_scd2 rows: 22

--- Customers that now have HISTORY (more than one version) ---
+-----------+--------------+---------+-----------+------------------------+----------+----------+----------+
|customer_id|name          |city     |segment    |email                   |start_date|end_date  |is_current|
+-----------+--------------+---------+-----------+------------------------+----------+----------+----------+
|C002       |Bhavna Mehta  |Pune     |Corporate  |bhavna.mehta@example.com|2024-01-10|2024-03-02|false     |
|C002       |Bhavna Mehta  |Bengaluru|Corporate  |bhavna.mehta@example.com|2024-03-02|NULL      |true      |
|C005       |Esha Khan     |Delhi    |Corporate  |esha.khan@example.com   |2024-01-12|2024-03-02|false     |
|C005       |Esha Khan     |Delhi    |Home Office|esha.khan@corp.com      |2024-03-02|NULL      |true      |
|C009       |Ishita Rao    |Unknown  |Home Office|ishita.rao@example.com  |2024-01-14|2024-03-03|false     |
|C009       |Ishita Rao    |Goa      |Ho

In [207]:
print("--- Current view (is_current = true) — this is what a BI tool would query ---")
(scd2_df.filter("is_current = true")
        .orderBy("customer_id")
        .select("customer_id", "name", "city", "segment", "start_date", "is_current")
        .show(50, truncate=False))

--- Current view (is_current = true) — this is what a BI tool would query ---
+-----------+--------------+----------+-----------+----------+----------+
|customer_id|name          |city      |segment    |start_date|is_current|
+-----------+--------------+----------+-----------+----------+----------+
|C001       |Aarav Sharma  |Mumbai    |Consumer   |2024-01-10|true      |
|C002       |Bhavna Mehta  |Bengaluru |Corporate  |2024-03-02|true      |
|C003       |Chirag Iyer   |Bengaluru |Consumer   |2024-01-11|true      |
|C004       |Divya Nair    |Kochi     |Home Office|2024-01-11|true      |
|C005       |Esha Khan     |Delhi     |Home Office|2024-03-02|true      |
|C006       |Farhan Ali    |Hyderabad |Consumer   |2024-01-12|true      |
|C007       |Gita Verma    |Jaipur    |Consumer   |2024-01-13|true      |
|C008       |Harsh Patel   |Ahmedabad |Corporate  |2024-01-13|true      |
|C009       |Ishita Rao    |Goa       |Home Office|2024-03-03|true      |
|C010       |Jatin Grover  |Chandi

In [208]:
# Point-in-time question: where did each customer live on 2024-02-15?
print("--- State of the dimension as of 2024-02-15 ---")
(scd2_df
 .filter("start_date <= '2024-02-15' AND (end_date IS NULL OR end_date > '2024-02-15')")
 .orderBy("customer_id")
 .select("customer_id", "name", "city", "segment")
 .show(50, truncate=False))

--- State of the dimension as of 2024-02-15 ---
+-----------+--------------+----------+-----------+
|customer_id|name          |city      |segment    |
+-----------+--------------+----------+-----------+
|C001       |Aarav Sharma  |Mumbai    |Consumer   |
|C002       |Bhavna Mehta  |Pune      |Corporate  |
|C003       |Chirag Iyer   |Bengaluru |Consumer   |
|C004       |Divya Nair    |Kochi     |Home Office|
|C005       |Esha Khan     |Delhi     |Corporate  |
|C006       |Farhan Ali    |Hyderabad |Consumer   |
|C007       |Gita Verma    |Jaipur    |Consumer   |
|C008       |Harsh Patel   |Ahmedabad |Corporate  |
|C009       |Ishita Rao    |Unknown   |Home Office|
|C010       |Jatin Grover  |Chandigarh|Consumer   |
|C011       |Kavya Menon   |Chennai   |Corporate  |
|C012       |Lakshay Bhatia|Meerut    |Consumer   |
|C013       |Meera Joshi   |Indore    |Home Office|
|C014       |Nikhil Bose   |Kolkata   |Consumer   |
|C015       |Ojas Kulkarni |Nagpur    |Corporate  |
+-----------+---

## 7. Bonus — `WHEN NOT MATCHED BY SOURCE`

Available in Delta 2.3+ / Databricks Runtime 12.2 LTS and above. It acts on **target rows
that the source did not mention at all** — the usual use is soft-deleting or flagging
records that disappeared from the upstream system.

A separate demo table is used so the SCD tables above stay intact.

In [209]:
# Build a small demo table with a 'status' column
demo = clean_master.withColumn("status", F.lit("active"))
demo.write.format("delta").mode("overwrite").save(spark_path(SYNC_PATH))
spark.sql(f"CREATE TABLE customer_sync_demo USING DELTA LOCATION '{spark_path(SYNC_PATH)}'")

demo_tbl = DeltaTable.forPath(spark, spark_path(SYNC_PATH))
print(f"Demo table rows: {demo_tbl.toDF().count()}")

Demo table rows: 15


In [210]:
(demo_tbl.alias("t")
 .merge(updates.alias("s"), "t.customer_id = s.customer_id")
 .whenMatchedUpdate(set={"city": "s.city", "segment": "s.segment",
                         "updated_at": "s.updated_at",
                         "status": F.lit("active")})
 .whenNotMatchedInsert(values={
     "customer_id": "s.customer_id", "name": "s.name", "email": "s.email",
     "city": "s.city", "segment": "s.segment", "phone": "s.phone",
     "updated_at": "s.updated_at", "status": F.lit("active")})
 # target rows the source never mentioned -> mark them dormant
 .whenNotMatchedBySourceUpdate(
     condition="t.status = 'active'",
     set={"status": F.lit("dormant")})
 .execute())

(spark.table("customer_sync_demo")
      .groupBy("status").count().show())

(spark.table("customer_sync_demo")
      .orderBy("status", "customer_id")
      .select("customer_id", "name", "city", "updated_at", "status")
      .show(50, truncate=False))

+-------+-----+
| status|count|
+-------+-----+
| active|    8|
|dormant|   10|
+-------+-----+

+-----------+--------------+----------+----------+-------+
|customer_id|name          |city      |updated_at|status |
+-----------+--------------+----------+----------+-------+
|C001       |Aarav Sharma  |Mumbai    |2024-03-04|active |
|C002       |Bhavna Mehta  |Bengaluru |2024-03-02|active |
|C005       |Esha Khan     |Delhi     |2024-03-02|active |
|C009       |Ishita Rao    |Goa       |2024-03-03|active |
|C012       |Lakshay Bhatia|Noida     |2024-03-03|active |
|C016       |Priya Deshmukh|Surat     |2024-03-04|active |
|C017       |Rahul Sinha   |Lucknow   |2024-03-05|active |
|C018       |Sneha Kapoor  |Bhopal    |2024-03-05|active |
|C003       |Chirag Iyer   |Bengaluru |2024-01-11|dormant|
|C004       |Divya Nair    |Kochi     |2024-01-11|dormant|
|C006       |Farhan Ali    |Hyderabad |2024-01-12|dormant|
|C007       |Gita Verma    |Jaipur    |2024-01-13|dormant|
|C008       |Harsh

## 8. Validation

Four checks: **row counts**, **no duplicate keys**, **exactly one current version per
customer in SCD2**, and **the transaction history / time travel**.

In [211]:
# --- 8.1 Row counts ----------------------------------------------------
counts = spark.createDataFrame(
    [
        ("Raw master CSV",              raw_master.count()),
        ("Cleaned master (Delta seed)", clean_master.count()),
        ("Raw incremental CSV",         raw_incr.count()),
        ("De-duplicated incremental",   updates.count()),
        ("customer_scd1 (final)",       spark.table("customer_scd1").count()),
        ("customer_scd2 (all versions)", spark.table("customer_scd2").count()),
        ("customer_scd2 (current only)",
         spark.table("customer_scd2").filter("is_current = true").count()),
    ],
    ["dataset", "row_count"],
)
try:
    counts.show(truncate=False)
except Exception as e:
    print(f"counts.show() failed ({type(e).__name__}). Falling back to plain print:")
    fallback_counts = [
        ("Raw master CSV", raw_master.count()),
        ("Cleaned master (Delta seed)", clean_master.count()),
        ("Raw incremental CSV", raw_incr.count()),
        ("De-duplicated incremental", updates.count()),
        ("customer_scd1 (final)", spark.table("customer_scd1").count()),
        ("customer_scd2 (all versions)", spark.table("customer_scd2").count()),
        ("customer_scd2 (current only)", spark.table("customer_scd2").filter("is_current = true").count()),
    ]
    for dataset, row_count in fallback_counts:
        print(f"{dataset:<35} {row_count}")

counts.show() failed (Py4JJavaError). Falling back to plain print:
Raw master CSV                      18
Cleaned master (Delta seed)         15
Raw incremental CSV                 10
De-duplicated incremental           8
customer_scd1 (final)               18
customer_scd2 (all versions)        22
customer_scd2 (current only)        18


In [212]:
# --- 8.2 Duplicate checks ---------------------------------------------
dup_scd1 = (spark.table("customer_scd1")
            .groupBy("customer_id").count().filter("count > 1"))
print(f"SCD1 duplicate customer_id : {dup_scd1.count()}")
dup_scd1.show()

dup_current = (spark.table("customer_scd2")
               .filter("is_current = true")
               .groupBy("customer_id").count().filter("count > 1"))
print(f"SCD2 customers with >1 CURRENT row : {dup_current.count()}")
dup_current.show()

open_ended = (spark.table("customer_scd2")
              .filter("is_current = true AND end_date IS NOT NULL").count())
closed_open = (spark.table("customer_scd2")
               .filter("is_current = false AND end_date IS NULL").count())
print(f"Current rows wrongly carrying an end_date : {open_ended}")
print(f"Closed rows missing an end_date           : {closed_open}")

SCD1 duplicate customer_id : 0
+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+

SCD2 customers with >1 CURRENT row : 0
+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+

Current rows wrongly carrying an end_date : 0
Closed rows missing an end_date           : 0


In [213]:
# --- 8.3 Assertions — the notebook fails loudly if anything is off ------
expected_scd1 = 18   # 15 cleaned master customers + 3 new from the incremental batch

assert dup_scd1.count() == 0, "SCD1 has duplicate keys"
assert dup_current.count() == 0, "SCD2 has more than one current row for a customer"
assert open_ended == 0 and closed_open == 0, "SCD2 flag/date columns are inconsistent"
assert spark.table("customer_scd1").count() == expected_scd1, "unexpected SCD1 row count"

# C002 must show the NEW city, not the late-arriving old one
c002 = spark.table("customer_scd1").filter("customer_id = 'C002'").first()
assert c002["city"] == "Bengaluru", f"late-arriving record won: {c002['city']}"

# C001 was a no-op -> it must still have exactly one version in SCD2
c001_versions = spark.table("customer_scd2").filter("customer_id = 'C001'").count()
assert c001_versions == 1, f"no-op record created a spurious version ({c001_versions})"

print("ALL VALIDATION CHECKS PASSED")

ALL VALIDATION CHECKS PASSED


In [214]:
# --- 8.4 Transaction history and time travel ---------------------------
print("=== customer_scd2 history ===")
(spark.sql("DESCRIBE HISTORY customer_scd2")
      .select("version", "timestamp", "operation")
      .orderBy("version").show(truncate=False))

# Read the table as it looked at version 0 (before any merge)
v0 = spark.read.format("delta").option("versionAsOf", 0).load(spark_path(SCD2_PATH))
print(f"Version 0 row count : {v0.count()}")
print(f"Latest  row count   : {spark.table('customer_scd2').count()}")
v0.filter("customer_id = 'C002'").select("customer_id", "city", "is_current").show()

=== customer_scd2 history ===
+-------+-----------------------+---------+
|version|timestamp              |operation|
+-------+-----------------------+---------+
|0      |2026-08-02 21:36:52.153|WRITE    |
|1      |2026-08-02 21:36:59.352|MERGE    |
+-------+-----------------------+---------+

Version 0 row count : 15
Latest  row count   : 22
+-----------+----+----------+
|customer_id|city|is_current|
+-----------+----+----------+
|       C002|Pune|      true|
+-----------+----+----------+



## 9. Final dataset and summary

In [215]:
print("=" * 70)
print("FINAL SCD TYPE 1 TABLE (customer_scd1)")
print("=" * 70)
spark.table("customer_scd1").orderBy("customer_id").show(50, truncate=False)

FINAL SCD TYPE 1 TABLE (customer_scd1)
+-----------+--------------+------------------------+----------+-----------+----------+----------+
|customer_id|name          |email                   |city      |segment    |phone     |updated_at|
+-----------+--------------+------------------------+----------+-----------+----------+----------+
|C001       |Aarav Sharma  |aarav.sharma@example.com|Mumbai    |Consumer   |9810000001|2024-03-04|
|C002       |Bhavna Mehta  |bhavna.mehta@example.com|Bengaluru |Corporate  |9810000002|2024-03-02|
|C003       |Chirag Iyer   |chirag.iyer@example.com |Bengaluru |Consumer   |9810000003|2024-01-11|
|C004       |Divya Nair    |divya.nair@example.com  |Kochi     |Home Office|9810000004|2024-01-11|
|C005       |Esha Khan     |esha.khan@corp.com      |Delhi     |Home Office|9820000005|2024-03-02|
|C006       |Farhan Ali    |farhan.ali@example.com  |Hyderabad |Consumer   |9810000006|2024-01-12|
|C007       |Gita Verma    |unknown@example.com     |Jaipur    |Consum

In [216]:
print("=" * 70)
print("FINAL SCD TYPE 2 TABLE (customer_scd2) — all versions")
print("=" * 70)
(spark.table("customer_scd2")
      .orderBy("customer_id", "start_date")
      .show(50, truncate=False))

FINAL SCD TYPE 2 TABLE (customer_scd2) — all versions
+-----------+--------------+------------------------+----------+-----------+----------+----------+----------+----------+----------+
|customer_id|name          |email                   |city      |segment    |phone     |updated_at|start_date|end_date  |is_current|
+-----------+--------------+------------------------+----------+-----------+----------+----------+----------+----------+----------+
|C001       |Aarav Sharma  |aarav.sharma@example.com|Mumbai    |Consumer   |9810000001|2024-01-10|2024-01-10|NULL      |true      |
|C002       |Bhavna Mehta  |bhavna.mehta@example.com|Pune      |Corporate  |9810000002|2024-01-10|2024-01-10|2024-03-02|false     |
|C002       |Bhavna Mehta  |bhavna.mehta@example.com|Bengaluru |Corporate  |9810000002|2024-03-02|2024-03-02|NULL      |true      |
|C003       |Chirag Iyer   |chirag.iyer@example.com |Bengaluru |Consumer   |9810000003|2024-01-11|2024-01-11|NULL      |true      |
|C004       |Divya Nai

In [217]:
import sys, types

# Export both tables to CSV so the results are easy to attach to the submission
scd1_out = OUTPUT_DIR / "customer_scd1_final.csv"
scd2_out = OUTPUT_DIR / "customer_scd2_final.csv"

# Python 3.12+ compatibility: provide a distutils shim for libs that still import it.
# setuptools is not guaranteed to be installed in the notebook kernel, so keep the
# fallback self-contained and only rely on the standard library.
try:
    import distutils  # noqa: F401
except ModuleNotFoundError:
    distutils = types.ModuleType("distutils")
    distutils.__path__ = []

    version_mod = types.ModuleType("distutils.version")

    class LooseVersion(str):
        pass

    version_mod.LooseVersion = LooseVersion
    version_mod.StrictVersion = LooseVersion

    distutils.version = version_mod
    sys.modules["distutils"] = distutils
    sys.modules["distutils.version"] = version_mod

spark.table("customer_scd1").orderBy("customer_id").toPandas().to_csv(scd1_out, index=False)
spark.table("customer_scd2").orderBy("customer_id", "start_date") \
     .toPandas().to_csv(scd2_out, index=False)

print(f"Saved -> {scd1_out}")
print(f"Saved -> {scd2_out}")

Saved -> c:\Users\hp\Desktop\CelebalAssignments\Assignment7\output\customer_scd1_final.csv
Saved -> c:\Users\hp\Desktop\CelebalAssignments\Assignment7\output\customer_scd2_final.csv


In [ ]:
n_new = updates.filter(~F.col("customer_id").isin(list(before_ids))).count()
n_upd = updates.count() - n_new

summary_rows = [
    ("Source rows in incremental batch (raw)", raw_incr.count()),
    ("Source rows after de-duplication", updates.count()),
    ("Records INSERTED (new customers)", n_new),
    ("Records MATCHED for update", n_upd),
    ("SCD1 final row count", spark.table("customer_scd1").count()),
    ("SCD2 total versions", spark.table("customer_scd2").count()),
    ("SCD2 current versions", spark.table("customer_scd2").filter("is_current = true").count()),
    ("SCD2 historical (closed) versions", spark.table("customer_scd2").filter("is_current = false").count()),
    ("Duplicate keys in SCD1", 0),
    ("Duplicate current rows in SCD2", 0),
]

print("Summary")
print("-" * 70)
for metric, value in summary_rows:
    print(f"{metric:<40} {value}")

Py4JJavaError: An error occurred while calling o4217.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 3490.0 failed 1 times, most recent failure: Lost task 0.0 in stage 3490.0 (TID 2507) (Kashish executor driver): org.apache.spark.SparkException: Python worker exited unexpectedly (crashed)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:612)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:594)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:38)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:789)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:766)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.hasNext(PythonRunner.scala:525)
	at org.apache.spark.InterruptibleIterator.hasNext(InterruptibleIterator.scala:37)
	at sc

### Short explanation (paste this into your report)

The customer master CSV was loaded into Spark, cleaned (null keys dropped, optional
attributes defaulted, whitespace trimmed, exact and key-level duplicates removed) and
written to two Delta tables — one for SCD Type 1 and one for SCD Type 2.

A second CSV simulated the next day's feed. It was de-duplicated with a
`row_number()` window ordered by `updated_at DESC`, because Delta refuses a `MERGE` where
two source rows match the same target row, and because that same window discards the
late-arriving record for `C002`.

**Type 1** used a single `MERGE`: `WHEN MATCHED AND s.updated_at > t.updated_at THEN UPDATE`
plus `WHEN NOT MATCHED THEN INSERT`. Existing customers were overwritten in place and the
three new customers were appended — 15 rows became 18, with no history retained.

**Type 2** used the staged-source pattern. Each incoming row was staged with
`mergeKey = customer_id` so it would match and *close* the active version, and rows whose
tracked attributes had genuinely changed were staged a second time with `mergeKey = NULL`
so they could only fall through to `WHEN NOT MATCHED` and be *inserted* as a new version.
Null-safe comparison (`<=>`) drives the change detection, so `C001` — identical to what was
already stored — correctly produced no new version.

Validation confirmed the expected row counts, zero duplicate keys in Type 1, exactly one
`is_current = true` row per customer in Type 2, consistent `end_date` / `is_current` flags,
and a full audit trail in `DESCRIBE HISTORY`, with version 0 still readable via time travel.

## 10. Screenshot checklist

Take these while the notebook is running and drop them into `screenshots/`:

| Folder | Capture |
|---|---|
| `data_loading/` | Section 1 output (`raw_master.show()`, schema) and Section 4 (`raw_incr.show()`) |
| `data_cleaning/` | Section 2 — the null-count table and the before/after row counts |
| `scd1/` | Section 5 — the merge cell and the `AFTER merge` table |
| `scd2/` | Section 6 — the staged source, and the "customers that now have HISTORY" output |
| `validation/` | Section 8 — the counts table, duplicate checks and `ALL VALIDATION CHECKS PASSED` |
| `final_output/` | Section 9 — both final tables and the summary table |

Windows: `Win + Shift + S`.  macOS: `Cmd + Shift + 4`.

In [ ]:
# Release the Spark session (frees the JVM and the port)
spark.stop()
print("Spark session stopped. Assignment 2 complete.")